In [ ]:
import pandas as pd
import src.load_data as load_data
import re

# Loading all dataframes
movies_df = load_data.load_movies()
releases_df = load_data.load_releases()
genres_df = load_data.load_genres()
countries_df = load_data.load_countries()
languages_df = load_data.load_languages()
themes_df = load_data.load_themes()
oscars_df = load_data.load_oscars()
reviews_df = load_data.load_reviews()

# Core Movies DataFrame Creation
core_movies_df = movies_df[['id', 'title', 'global_release_year']].copy()

# Earliest Release Date and Month
earliest_releases = releases_df.loc[releases_df.groupby('id')['release_date'].idxmin()][['id', 'release_date']].copy()
earliest_releases.rename(columns={'release_date': 'earliest_release_date'}, inplace=True)
core_movies_df = pd.merge(core_movies_df, earliest_releases, on='id', how='left')
core_movies_df['earliest_release_month'] = core_movies_df['earliest_release_date'].dt.month.astype('Int64')

# Genres Aggregation
genres_aggregated = genres_df.groupby('id')['genre_category'].apply(list).reset_index()
genres_aggregated.rename(columns={'genre_category': 'genres_list'}, inplace=True)
core_movies_df = pd.merge(core_movies_df, genres_aggregated, on='id', how='left')

# Countries Aggregation
countries_aggregated = countries_df.groupby('id')['production_country'].apply(list).reset_index()
countries_aggregated.rename(columns={'production_country': 'production_countries_list'}, inplace=True)
core_movies_df = pd.merge(core_movies_df, countries_aggregated, on='id', how='left')

# Primary Language
primary_languages = languages_df[languages_df['language_type'].str.contains('primary', case=False, na=False)].copy()
primary_languages = primary_languages.drop_duplicates(subset=['id'], keep='first')[['id', 'language']]
primary_languages.rename(columns={'language': 'primary_language'}, inplace=True)
core_movies_df = pd.merge(core_movies_df, primary_languages, on='id', how='left')
core_movies_df['is_primary_language_english'] = core_movies_df['primary_language'].str.lower() == 'english'

# Oscar Data Integration

# Clean titles for matching
def clean_title_for_matching(title):
    if pd.isna(title):
        return None
    title = str(title).lower()
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

oscars_df['film_cleaned'] = oscars_df['film'].apply(clean_title_for_matching)
oscars_df['year_film'] = pd.to_numeric(oscars_df['year_film'], errors='coerce').astype('Int64')
core_movies_df['title_cleaned'] = core_movies_df['title'].apply(clean_title_for_matching)

# Merging Oscar nominations/wins
movies_with_oscars_info = pd.merge(
    core_movies_df.dropna(subset=['title_cleaned', 'global_release_year']),
    oscars_df.dropna(subset=['film_cleaned', 'year_film']),
    left_on=['title_cleaned', 'global_release_year'],
    right_on=['film_cleaned', 'year_film'],
    how='left',
    suffixes=('', '_oscar')
)

# Defining precise category lists
foreign_international_categories_exact = ["foreign language film", "honorary foreign language film award", "international feature film", "special foreign language film award"]

movies_with_oscars_info['category_lower'] = movies_with_oscars_info['category'].astype(str).str.lower()

movies_with_oscars_info['is_foreign_international_nom'] = movies_with_oscars_info['category_lower'].isin(foreign_international_categories_exact)
movies_with_oscars_info['is_foreign_international_win'] = movies_with_oscars_info['is_foreign_international_nom'] & movies_with_oscars_info['winner'].fillna(False)

oscar_summary = movies_with_oscars_info.groupby('id').agg(
    num_oscar_nominations=('category', 'count'),
    num_oscar_wins=('winner', 'sum'),
    foreign_international_nominated=('is_foreign_international_nom', 'max'),
    foreign_international_won=('is_foreign_international_win', 'max'),
).reset_index()

core_movies_df = pd.merge(core_movies_df, oscar_summary, on='id', how='left')

# Fill NaNs for Oscar columns
cols_to_fill_zero = ['num_oscar_nominations', 'num_oscar_wins']
core_movies_df[cols_to_fill_zero] = core_movies_df[cols_to_fill_zero].fillna(0).astype(int)

cols_to_fill_false = ['foreign_international_nominated', 'foreign_international_won']
core_movies_df[cols_to_fill_false] = core_movies_df[cols_to_fill_false].fillna(False).astype(bool)

# Dropping columns not needed anymore
core_movies_df = core_movies_df.drop(columns=['primary_language', 'title_cleaned'])

# Filter out movies with global_release_year > 2023
core_movies_df = core_movies_df[core_movies_df['global_release_year'] <= 2023]

print(core_movies_df.info())
print(core_movies_df.head())

# Save the DataFrame
core_movies_df.to_parquet("parquet-data/core_movies_df_processed.parquet")
print("Core movies DataFrame saved to parquet-data/core_movies_df_processed.parquet")

<class 'pandas.core.frame.DataFrame'>
Index: 830290 entries, 0 to 940970
Data columns (total 12 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   id                               830290 non-null  int64         
 1   title                            830288 non-null  string        
 2   global_release_year              830290 non-null  Int64         
 3   earliest_release_date            806912 non-null  datetime64[ns]
 4   earliest_release_month           806912 non-null  Int64         
 5   genres_list                      631831 non-null  object        
 6   production_countries_list        576614 non-null  object        
 7   is_primary_language_english      830290 non-null  bool          
 8   num_oscar_nominations            830290 non-null  int64         
 9   num_oscar_wins                   830290 non-null  int64         
 10  foreign_international_nominated  830290 non-null 

/var/folders/p3/7j0854c11kbc79b9lhv0_2j00000gn/T/ipykernel_63379/3025033460.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  core_movies_df[cols_to_fill_false] = core_movies_df[cols_to_fill_false].fillna(False).astype(bool)
